# 67. The Kitting & Value-Added Service Scheduling Problem
## Tier 3: The Advanced Algorithm (Metaheuristic Implementation)

### Key Assumptions
- Population-based optimization using genetic algorithm principles
- Multi-objective optimization balancing cost, service level, and efficiency
- Evolutionary search with crossover, mutation, and selection operators
- Adaptive parameter tuning for convergence optimization

### Approach (Step-by-Step)
1. **Chromosome Encoding**: Represent kitting schedule as 3D array (kits × shifts × periods)
2. **Population Initialization**: Generate diverse initial solutions with random valid schedules
3. **Fitness Evaluation**: Multi-objective fitness function considering cost, service, and utilization
4. **Genetic Operations**: Tournament selection, uniform crossover, and adaptive mutation
5. **Evolution Loop**: Iteratively improve population through selection and variation
6. **Convergence Analysis**: Track fitness improvement and diversity metrics

### What to Look For in the Results
- Population evolution and fitness improvement over generations
- Convergence behavior and solution diversity analysis
- Multi-objective optimization performance vs heuristic methods
- Trade-offs between different objectives (cost vs service level)

### Concrete Example (from the source)
We'll implement the genetic algorithm with 5 kits, 3 shifts, and 7 periods:
- Initial population: 50 diverse solutions
- Evolution: 500 generations with elitism preservation
- Target: 23% cost reduction vs greedy heuristic
- Expected service level: 96.5% vs 89.2% for heuristic

In [ ]:
# Import required libraries for genetic algorithm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
import copy
from collections import deque

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
@dataclass
class GAProblem:
    """Data structure for genetic algorithm kitting problem"""
    num_kits: int
    num_shifts: int
    num_periods: int
    
    # Problem parameters
    processing_times: np.ndarray  # [kit]
    cost_savings: np.ndarray     # [kit]
    demand: np.ndarray          # [kit][period]
    capacities: np.ndarray      # [shift][period]
    labor_costs: np.ndarray     # [shift][period]
    holding_costs: np.ndarray   # [kit][period]
    shortage_penalty: float = 10.0
    
    # GA parameters
    population_size: int = 50
    max_generations: int = 500
    crossover_rate: float = 0.8
    mutation_rate: float = 0.1
    tournament_size: int = 3
    elitism_ratio: float = 0.1

@dataclass
class Chromosome:
    """Represents a complete kitting schedule solution"""
    genes: np.ndarray  # 3D array: [kit][shift][period]
    fitness: float = 0.0
    rank: int = 0
    crowding_distance: float = 0.0
    
    def copy(self):
        """Create a deep copy of the chromosome"""
        return Chromosome(
            genes=self.genes.copy(),
            fitness=self.fitness,
            rank=self.rank,
            crowding_distance=self.crowding_distance
        )

def create_ga_example():
    """Create the example problem from the source material"""
    return GAProblem(
        num_kits=5,
        num_shifts=3,
        num_periods=7,
        
        # Processing times per kit (minutes)
        processing_times=np.array([3, 5, 4, 6, 7]),
        
        # Cost savings per kit ($)
        cost_savings=np.array([2.50, 4.00, 3.20, 5.50, 4.80]),
        
        # Demand matrix [kit][period]
        demand=np.array([
            [45, 50, 40, 55, 48, 52, 47],  # Kit 1
            [30, 35, 32, 38, 34, 36, 33],  # Kit 2
            [25, 28, 26, 30, 27, 29, 25],  # Kit 3
            [20, 22, 21, 24, 23, 25, 22],  # Kit 4
            [15, 18, 16, 19, 17, 20, 18],   # Kit 5
        ]),
        
        # Capacity matrix [shift][period]
        capacities=np.array([
            [480, 480, 480, 480, 480, 480, 480],  # Shift 1
            [420, 420, 420, 420, 420, 420, 420],  # Shift 2
            [360, 360, 360, 360, 360, 360, 360],  # Shift 3
        ]),
        
        # Labor costs [shift][period]
        labor_costs=np.array([
            [15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0],  # Shift 1
            [18.0, 18.0, 18.0, 18.0, 18.0, 18.0, 18.0],  # Shift 2
            [22.0, 22.0, 22.0, 22.0, 22.0, 22.0, 22.0],  # Shift 3
        ]),
        
        # Holding costs [kit][period]
        holding_costs=np.array([
            [0.80, 0.80, 0.80, 0.80, 0.80, 0.80, 0.80],
            [1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00],
            [0.90, 0.90, 0.90, 0.90, 0.90, 0.90, 0.90],
            [1.20, 1.20, 1.20, 1.20, 1.20, 1.20, 1.20],
            [1.10, 1.10, 1.10, 1.10, 1.10, 1.10, 1.10],
        ]),
        
        shortage_penalty=10.0,
        population_size=50,
        max_generations=500,
        crossover_rate=0.8,
        mutation_rate=0.1,
        tournament_size=3,
        elitism_ratio=0.1
    )

# Create the problem instance
problem = create_ga_example()
print(f"GA problem created: {problem.num_kits} kits, {problem.num_shifts} shifts, {problem.num_periods} periods")
print(f"Population size: {problem.population_size}, Max generations: {problem.max_generations}")

In [ ]:
def generate_random_chromosome(problem: GAProblem) -> Chromosome:
    """Generate a random valid chromosome"""
    
    genes = np.zeros((problem.num_kits, problem.num_shifts, problem.num_periods))
    remaining_demand = problem.demand.copy()
    remaining_capacity = problem.capacities.copy()
    
    # Randomly assign production while respecting constraints
    for k in range(problem.num_kits):
        for t in range(problem.num_periods):
            if remaining_demand[k, t] > 0:
                # Randomly choose shifts for this kit and period
                available_shifts = [s for s in range(problem.num_shifts) 
                                  if remaining_capacity[s, t] >= problem.processing_times[k]]
                
                if available_shifts:
                    # Distribute demand across available shifts
                    np.random.shuffle(available_shifts)
                    
                    for s in available_shifts:
                        if remaining_demand[k, t] <= 0:
                            break
                        
                        max_producible = min(
                            remaining_demand[k, t],
                            int(remaining_capacity[s, t] / problem.processing_times[k])
                        )
                        
                        if max_producible > 0:
                            # Randomly decide how much to produce
                            produce = np.random.randint(0, max_producible + 1)
                            genes[k, s, t] = produce
                            remaining_demand[k, t] -= produce
                            remaining_capacity[s, t] -= produce * problem.processing_times[k]
    
    return Chromosome(genes=genes)

def initialize_population(problem: GAProblem) -> List[Chromosome]:
    """Initialize the population with diverse solutions"""
    
    population = []
    
    # Generate random chromosomes
    for i in range(problem.population_size):
        chromosome = generate_random_chromosome(problem)
        population.append(chromosome)
    
    return population

# Initialize population
population = initialize_population(problem)
print(f"Population initialized with {len(population)} chromosomes")

# Display sample chromosome statistics
sample = population[0]
total_production = np.sum(sample.genes)
print(f"Sample chromosome total production: {total_production:.0f} units")
print(f"Sample chromosome shape: {sample.genes.shape}")

In [ ]:
def calculate_multi_objective_fitness(chromosome: Chromosome, problem: GAProblem) -> Dict[str, float]:
    """Calculate multi-objective fitness for a chromosome"""
    
    genes = chromosome.genes
    
    # 1. Cost objective (minimize)
    production_cost = 0
    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                if genes[k, s, t] > 0:
                    production_cost += (genes[k, s, t] * problem.processing_times[k] * 
                                      problem.labor_costs[s, t] / 60)
    
    # Inventory cost (simplified - assume 1 period holding)
    inventory_cost = np.sum(genes * problem.holding_costs[:, np.newaxis, :])
    
    # Shortage cost
    total_production_per_kit = np.sum(genes, axis=(1, 2))  # Sum over shifts and periods
    total_demand_per_kit = np.sum(problem.demand, axis=1)
    shortages = np.maximum(0, total_demand_per_kit - total_production_per_kit)
    shortage_cost = np.sum(shortages * problem.shortage_penalty)
    
    total_cost = production_cost + inventory_cost + shortage_cost
    
    # 2. Service level objective (maximize)
    total_demand = np.sum(problem.demand)
    total_production = np.sum(genes)
    service_level = (total_production / total_demand) * 100 if total_demand > 0 else 0
    
    # 3. Capacity utilization objective (maximize, but not too high)
    total_capacity = np.sum(problem.capacities)
    used_capacity = 0
    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                used_capacity += genes[k, s, t] * problem.processing_times[k]
    
    capacity_utilization = (used_capacity / total_capacity) * 100 if total_capacity > 0 else 0
    
    # Penalize very high utilization (>95%)
    if capacity_utilization > 95:
        utilization_penalty = (capacity_utilization - 95) * 2
        capacity_utilization -= utilization_penalty
    
    # 4. Load balancing objective (maximize)
    # Calculate coefficient of variation of capacity utilization across shifts
    shift_utilizations = []
    for s in range(problem.num_shifts):
        shift_used = 0
        for k in range(problem.num_kits):
            for t in range(problem.num_periods):
                shift_used += genes[k, s, t] * problem.processing_times[k]
        
        shift_total_capacity = np.sum(problem.capacities[s])
        shift_util = (shift_used / shift_total_capacity) * 100 if shift_total_capacity > 0 else 0
        shift_utilizations.append(shift_util)
    
    if len(shift_utilizations) > 1:
        mean_util = np.mean(shift_utilizations)
        std_util = np.std(shift_utilizations)
        cv = std_util / mean_util if mean_util > 0 else 0
        load_balancing = max(0, 100 - cv * 100)  # Convert to 0-100 scale
    else:
        load_balancing = 100
    
    return {
        'total_cost': total_cost,
        'service_level': service_level,
        'capacity_utilization': capacity_utilization,
        'load_balancing': load_balancing,
        'total_production': total_production
    }

def calculate_aggregated_fitness(objectives: Dict[str, float], weights: Optional[Dict[str, float]] = None) -> float:
    """Calculate aggregated fitness from multiple objectives"""
    
    if weights is None:
        # Default weights (can be adjusted based on priorities)
        weights = {
            'total_cost': 0.4,        # Minimize cost
            'service_level': 0.3,    # Maximize service
            'capacity_utilization': 0.2,  # Maximize utilization
            'load_balancing': 0.1    # Maximize balance
        }
    
    # Normalize objectives (cost is minimized, others are maximized)
    normalized_cost = 1.0 / (1.0 + objectives['total_cost'] / 1000)  # Scale and invert
    normalized_service = objectives['service_level'] / 100
    normalized_utilization = objectives['capacity_utilization'] / 100
    normalized_balance = objectives['load_balancing'] / 100
    
    # Calculate weighted sum
    fitness = (weights['total_cost'] * normalized_cost +
               weights['service_level'] * normalized_service +
               weights['capacity_utilization'] * normalized_utilization +
               weights['load_balancing'] * normalized_balance)
    
    return fitness

# Test fitness calculation on sample chromosome
sample_objectives = calculate_multi_objective_fitness(sample, problem)
sample_fitness = calculate_aggregated_fitness(sample_objectives)

print("Sample chromosome objectives:")
for key, value in sample_objectives.items():
    if key == 'total_cost':
        print(f"  {key}: ${value:.2f}")
    else:
        print(f"  {key}: {value:.1f}%")

print(f"Sample aggregated fitness: {sample_fitness:.4f}")

In [ ]:
def tournament_selection(population: List[Chromosome], tournament_size: int) -> Chromosome:
    """Tournament selection operator"""
    
    # Randomly select tournament participants
    tournament = random.sample(population, min(tournament_size, len(population)))
    
    # Return the best (highest fitness) participant
    best = max(tournament, key=lambda x: x.fitness)
    
    return best.copy()

def uniform_crossover(parent1: Chromosome, parent2: Chromosome, crossover_rate: float) -> Tuple[Chromosome, Chromosome]:
    """Uniform crossover operator"""
    
    if random.random() > crossover_rate:
        return parent1.copy(), parent2.copy()
    
    # Create offspring
    offspring1_genes = np.zeros_like(parent1.genes)
    offspring2_genes = np.zeros_like(parent2.genes)
    
    # Uniform crossover for each gene
    for k in range(parent1.genes.shape[0]):
        for s in range(parent1.genes.shape[1]):
            for t in range(parent1.genes.shape[2]):
                if random.random() < 0.5:
                    offspring1_genes[k, s, t] = parent1.genes[k, s, t]
                    offspring2_genes[k, s, t] = parent2.genes[k, s, t]
                else:
                    offspring1_genes[k, s, t] = parent2.genes[k, s, t]
                    offspring2_genes[k, s, t] = parent1.genes[k, s, t]
    
    offspring1 = Chromosome(genes=offspring1_genes)
    offspring2 = Chromosome(genes=offspring2_genes)
    
    return offspring1, offspring2

def adaptive_mutation(chromosome: Chromosome, problem: GAProblem, mutation_rate: float) -> Chromosome:
    """Adaptive mutation operator"""
    
    mutated = chromosome.copy()
    
    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                if random.random() < mutation_rate:
                    # Apply mutation
                    current_value = mutated.genes[k, s, t]
                    
                    # Mutation strength based on current value
                    if current_value > 0:
                        # Either increase, decrease, or set to zero
                        mutation_type = random.choice(['increase', 'decrease', 'zero'])
                        
                        if mutation_type == 'increase':
                            # Increase by small amount
                            increase = random.randint(1, max(1, int(current_value * 0.3)))
                            mutated.genes[k, s, t] += increase
                        elif mutation_type == 'decrease':
                            # Decrease but not below zero
                            decrease = random.randint(1, int(current_value * 0.5))
                            mutated.genes[k, s, t] = max(0, current_value - decrease)
                        else:  # zero
                            mutated.genes[k, s, t] = 0
                    else:
                        # Set to small random value
                        mutated.genes[k, s, t] = random.randint(1, 5)
    
    return mutated

def repair_chromosome(chromosome: Chromosome, problem: GAProblem) -> Chromosome:
    """Repair chromosome to satisfy constraints"""
    
    repaired = chromosome.copy()
    
    # Ensure production doesn't exceed demand
    for k in range(problem.num_kits):
        for t in range(problem.num_periods):
            total_production = np.sum(repaired.genes[k, :, t])
            if total_production > problem.demand[k, t]:
                # Scale down production proportionally
                scale_factor = problem.demand[k, t] / total_production
                repaired.genes[k, :, t] *= scale_factor
                repaired.genes[k, :, t] = np.round(repaired.genes[k, :, t])
    
    # Ensure capacity constraints are satisfied
    for s in range(problem.num_shifts):
        for t in range(problem.num_periods):
            capacity_used = 0
            for k in range(problem.num_kits):
                capacity_used += repaired.genes[k, s, t] * problem.processing_times[k]
            
            if capacity_used > problem.capacities[s, t]:
                # Scale down production to meet capacity
                scale_factor = problem.capacities[s, t] / capacity_used
                for k in range(problem.num_kits):
                    repaired.genes[k, s, t] *= scale_factor
                    repaired.genes[k, s, t] = np.round(repaired.genes[k, s, t])
    
    return repaired

# Test genetic operators
print("Testing genetic operators...")

# Test selection
selected = tournament_selection(population, problem.tournament_size)
print(f"Tournament selection completed, fitness: {selected.fitness:.4f}")

# Test crossover
parent1, parent2 = population[0], population[1]
offspring1, offspring2 = uniform_crossover(parent1, parent2, problem.crossover_rate)
print(f"Crossover completed, offspring shapes: {offspring1.genes.shape}, {offspring2.genes.shape}")

# Test mutation
mutated = adaptive_mutation(offspring1, problem, problem.mutation_rate)
print(f"Mutation completed")

# Test repair
repaired = repair_chromosome(mutated, problem)
print(f"Repair completed")

In [ ]:
def genetic_algorithm(problem: GAProblem) -> Dict:
    """Main genetic algorithm implementation"""
    
    start_time = time.time()
    
    # Initialize population
    population = initialize_population(problem)
    
    # Evaluate initial population
    for chromosome in population:
        objectives = calculate_multi_objective_fitness(chromosome, problem)
        chromosome.fitness = calculate_aggregated_fitness(objectives)
    
    # Sort by fitness
    population.sort(key=lambda x: x.fitness, reverse=True)
    
    # Track evolution statistics
    evolution_history = {
        'best_fitness': [],
        'avg_fitness': [],
        'best_objectives': [],
        'diversity': []
    }
    
    # Evolution loop
    for generation in range(problem.max_generations):
        
        # Elitism: preserve best chromosomes
        elite_size = int(problem.population_size * problem.elitism_ratio)
        new_population = population[:elite_size]
        
        # Generate offspring
        while len(new_population) < problem.population_size:
            
            # Selection
            parent1 = tournament_selection(population, problem.tournament_size)
            parent2 = tournament_selection(population, problem.tournament_size)
            
            # Crossover
            offspring1, offspring2 = uniform_crossover(parent1, parent2, problem.crossover_rate)
            
            # Mutation
            offspring1 = adaptive_mutation(offspring1, problem, problem.mutation_rate)
            offspring2 = adaptive_mutation(offspring2, problem, problem.mutation_rate)
            
            # Repair
            offspring1 = repair_chromosome(offspring1, problem)
            offspring2 = repair_chromosome(offspring2, problem)
            
            # Evaluate fitness
            objectives1 = calculate_multi_objective_fitness(offspring1, problem)
            objectives2 = calculate_multi_objective_fitness(offspring2, problem)
            offspring1.fitness = calculate_aggregated_fitness(objectives1)
            offspring2.fitness = calculate_aggregated_fitness(objectives2)
            
            # Add to new population
            new_population.extend([offspring1, offspring2])
        
        # Trim to exact population size
        new_population = new_population[:problem.population_size]
        
        # Replace old population
        population = new_population
        
        # Sort by fitness
        population.sort(key=lambda x: x.fitness, reverse=True)
        
        # Record statistics
        best_chromosome = population[0]
        avg_fitness = np.mean([c.fitness for c in population])
        
        # Calculate diversity (average pairwise distance)
        diversity = 0
        for i in range(len(population)):
            for j in range(i+1, len(population)):
                distance = np.sum(np.abs(population[i].genes - population[j].genes))
                diversity += distance
        
        if len(population) > 1:
            diversity /= (len(population) * (len(population) - 1) / 2)
        
        evolution_history['best_fitness'].append(best_chromosome.fitness)
        evolution_history['avg_fitness'].append(avg_fitness)
        evolution_history['best_objectives'].append(calculate_multi_objective_fitness(best_chromosome, problem))
        evolution_history['diversity'].append(diversity)
        
        # Progress reporting
        if generation % 50 == 0:
            print(f"Generation {generation:4d}: Best Fitness = {best_chromosome.fitness:.4f}, "
                  f"Avg Fitness = {avg_fitness:.4f}, Diversity = {diversity:.1f}")
    
    # Get final best solution
    best_chromosome = population[0]
    final_objectives = calculate_multi_objective_fitness(best_chromosome, problem)
    
    computation_time = time.time() - start_time
    
    return {
        'best_chromosome': best_chromosome,
        'best_objectives': final_objectives,
        'evolution_history': evolution_history,
        'computation_time': computation_time,
        'final_population': population
    }

# Run the genetic algorithm
print("Starting Genetic Algorithm...")
print("=" * 50)
ga_results = genetic_algorithm(problem)

print("\n" + "=" * 50)
print("GENETIC ALGORITHM COMPLETED")
print("=" * 50)
print(f"Computation time: {ga_results['computation_time']:.2f} seconds")
print(f"Best fitness: {ga_results['best_chromosome'].fitness:.4f}")

print("\nBest solution objectives:")
for key, value in ga_results['best_objectives'].items():
    if key == 'total_cost':
        print(f"  {key}: ${value:.2f}")
    else:
        print(f"  {key}: {value:.1f}%")

In [ ]:
def analyze_ga_results(ga_results: Dict, problem: GAProblem):
    """Comprehensive analysis of genetic algorithm results"""
    
    print("=" * 70)
    print("GENETIC ALGORITHM SOLUTION ANALYSIS")
    print("=" * 70)
    
    best_chromosome = ga_results['best_chromosome']
    objectives = ga_results['best_objectives']
    history = ga_results['evolution_history']
    
    # 1. Production Schedule Analysis
    print("\n1. OPTIMAL PRODUCTION SCHEDULE:")
    print("-" * 45)
    
    for k in range(problem.num_kits):
        print(f"\nKit {k+1} (Processing: {problem.processing_times[k]} min, Savings: ${problem.cost_savings[k]}):")
        
        total_kit_production = 0
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                quantity = best_chromosome.genes[k, s, t]
                if quantity > 0:
                    print(f"  Shift {s+1}, Period {t+1}: {quantity:.0f} units")
                    total_kit_production += quantity
        
        demand_kit = np.sum(problem.demand[k])
        fulfillment = (total_kit_production / demand_kit) * 100 if demand_kit > 0 else 0
        print(f"  Total: {total_kit_production:.0f}/{demand_kit:.0f} units ({fulfillment:.1f}% fulfillment)")
    
    # 2. Performance Metrics
    print("\n2. PERFORMANCE METRICS:")
    print("-" * 30)
    
    print(f"Total production: {objectives['total_production']:.0f} units")
    print(f"Total demand: {np.sum(problem.demand):.0f} units")
    print(f"Service level: {objectives['service_level']:.1f}%")
    print(f"Capacity utilization: {objectives['capacity_utilization']:.1f}%")
    print(f"Load balancing: {objectives['load_balancing']:.1f}%")
    print(f"Total cost: ${objectives['total_cost']:.2f}")
    print(f"Computation time: {ga_results['computation_time']:.2f} seconds")
    
    # 3. Evolution Analysis
    print("\n3. EVOLUTION ANALYSIS:")
    print("-" * 25)
    
    initial_fitness = history['best_fitness'][0]
    final_fitness = history['best_fitness'][-1]
    improvement = ((final_fitness - initial_fitness) / initial_fitness) * 100
    
    initial_avg = history['avg_fitness'][0]
    final_avg = history['avg_fitness'][-1]
    avg_improvement = ((final_avg - initial_avg) / initial_avg) * 100
    
    print(f"Initial best fitness: {initial_fitness:.4f}")
    print(f"Final best fitness: {final_fitness:.4f}")
    print(f"Best fitness improvement: {improvement:.1f}%")
    print(f"Average fitness improvement: {avg_improvement:.1f}%")
    
    # Find convergence generation (when improvement < 1% for 50 generations)
    convergence_gen = problem.max_generations
    for gen in range(50, len(history['best_fitness'])):
        recent_improvement = ((history['best_fitness'][gen] - history['best_fitness'][gen-50]) / 
                              history['best_fitness'][gen-50]) * 100
        if recent_improvement < 1.0:
            convergence_gen = gen
            break
    
    print(f"Convergence generation: {convergence_gen} (out of {problem.max_generations})")
    
    # 4. Capacity Utilization Details
    print("\n4. CAPACITY UTILIZATION DETAILS:")
    print("-" * 35)
    
    for s in range(problem.num_shifts):
        print(f"\nShift {s+1}:")
        
        for t in range(problem.num_periods):
            used_capacity = 0
            for k in range(problem.num_kits):
                used_capacity += best_chromosome.genes[k, s, t] * problem.processing_times[k]
            
            total_capacity = problem.capacities[s, t]
            utilization = (used_capacity / total_capacity) * 100 if total_capacity > 0 else 0
            
            print(f"  Period {t+1}: {used_capacity:.1f}/{total_capacity:.1f} min ({utilization:.1f}%)")
        
        # Shift summary
        shift_total_used = 0
        shift_total_capacity = 0
        for t in range(problem.num_periods):
            for k in range(problem.num_kits):
                shift_total_used += best_chromosome.genes[k, s, t] * problem.processing_times[k]
            shift_total_capacity += problem.capacities[s, t]
        
        shift_util = (shift_total_used / shift_total_capacity) * 100 if shift_total_capacity > 0 else 0
        print(f"  Shift total: {shift_util:.1f}% utilization")

# Analyze the GA results
analyze_ga_results(ga_results, problem)

In [ ]:
def create_ga_visualization(ga_results: Dict, problem: GAProblem):
    """Create comprehensive visualization of genetic algorithm results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Genetic Algorithm - Kitting Optimization Results', fontsize=16, fontweight='bold')
    
    history = ga_results['evolution_history']
    best_chromosome = ga_results['best_chromosome']
    
    # 1. Fitness Evolution
    ax1 = axes[0, 0]
    
    generations = range(len(history['best_fitness']))
    ax1.plot(generations, history['best_fitness'], 'b-', linewidth=2, label='Best Fitness')
    ax1.plot(generations, history['avg_fitness'], 'r--', linewidth=1.5, label='Average Fitness')
    ax1.set_title('Fitness Evolution Over Generations')
    ax1.set_xlabel('Generation')
    ax1.set_ylabel('Fitness')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Add convergence point
    convergence_gen = len(history['best_fitness'])
    for gen in range(50, len(history['best_fitness'])):
        recent_improvement = ((history['best_fitness'][gen] - history['best_fitness'][gen-50]) / 
                              history['best_fitness'][gen-50]) * 100
        if recent_improvement < 1.0:
            convergence_gen = gen
            break
    
    ax1.axvline(x=convergence_gen, color='green', linestyle=':', alpha=0.7, label=f'Convergence (Gen {convergence_gen})')
    ax1.legend()
    
    # 2. Population Diversity
    ax2 = axes[0, 1]
    
    ax2.plot(generations, history['diversity'], 'g-', linewidth=2)
    ax2.set_title('Population Diversity Over Generations')
    ax2.set_xlabel('Generation')
    ax2.set_ylabel('Average Pairwise Distance')
    ax2.grid(True, alpha=0.3)
    
    # Highlight diversity loss
    if len(history['diversity']) > 100:
        initial_diversity = np.mean(history['diversity'][:10])
        final_diversity = np.mean(history['diversity'][-10:])
        diversity_loss = ((initial_diversity - final_diversity) / initial_diversity) * 100
        ax2.text(0.05, 0.95, f'Diversity Loss: {diversity_loss:.1f}%', 
                transform=ax2.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # 3. Multi-Objective Performance
    ax3 = axes[1, 0]
    
    # Track objective evolution
    cost_evolution = [obj['total_cost'] for obj in history['best_objectives']]
    service_evolution = [obj['service_level'] for obj in history['best_objectives']]
    utilization_evolution = [obj['capacity_utilization'] for obj in history['best_objectives']]
    
    # Normalize for visualization
    cost_norm = [(c - min(cost_evolution)) / (max(cost_evolution) - min(cost_evolution)) for c in cost_evolution]
    service_norm = [s / 100 for s in service_evolution]
    util_norm = [u / 100 for u in utilization_evolution]
    
    ax3.plot(generations, cost_norm, 'r-', linewidth=2, label='Cost (normalized)')
    ax3.plot(generations, service_norm, 'b-', linewidth=2, label='Service Level')
    ax3.plot(generations, util_norm, 'g-', linewidth=2, label='Capacity Utilization')
    ax3.set_title('Multi-Objective Evolution')
    ax3.set_xlabel('Generation')
    ax3.set_ylabel('Normalized Value')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 1)
    
    # 4. Production Schedule Heatmap
    ax4 = axes[1, 1]
    
    # Aggregate production by kit and shift
    production_matrix = np.zeros((problem.num_kits, problem.num_shifts))
    
    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            production_matrix[k, s] = np.sum(best_chromosome.genes[k, s, :])
    
    sns.heatmap(production_matrix, 
                annot=True, fmt='.0f', cmap='YlOrRd',
                xticklabels=[f'Shift {s+1}' for s in range(problem.num_shifts)],
                yticklabels=[f'Kit {k+1}' for k in range(problem.num_kits)],
                ax=ax4)
    ax4.set_title('Production Schedule (Total Units per Kit-Shift)')
    ax4.set_xlabel('Shift')
    ax4.set_ylabel('Kit')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualization
fig = create_ga_visualization(ga_results, problem)

In [ ]:
def compare_with_baselines():
    """Compare GA performance with baseline methods"""
    
    print("=" * 60)
    print("PERFORMANCE COMPARISON WITH BASELINES")
    print("=" * 60)

    # Simple greedy baseline (similar to Tier 2 heuristic)
    def greedy_baseline(problem):
        """Simple greedy assignment for comparison"""
        genes = np.zeros((problem.num_kits, problem.num_shifts, problem.num_periods))
        
        # Priority: highest cost savings per processing time first
        priorities = problem.cost_savings / problem.processing_times
        kit_order = np.argsort(priorities)[::-1]  # Descending order
        
        for k in kit_order:
            for t in range(problem.num_periods):
                if problem.demand[k, t] > 0:
                    # Find shift with most available capacity
                    best_shift = 0
                    best_capacity = 0
                    
                    for s in range(problem.num_shifts):
                        available = problem.capacities[s, t]
                        if available > best_capacity:
                            best_capacity = available
                            best_shift = s
                    
                    # Assign as much as possible
                    max_producible = min(
                        problem.demand[k, t],
                        int(best_capacity / problem.processing_times[k])
                    )
                    
                    genes[k, best_shift, t] = max_producible
        
        return Chromosome(genes=genes)
    
    # Random assignment baseline
    def random_baseline(problem):
        """Random assignment for comparison"""
        return generate_random_chromosome(problem)
    
    # Run baselines
    print("Running baseline comparisons...")
    
    greedy_solution = greedy_baseline(problem)
    greedy_objectives = calculate_multi_objective_fitness(greedy_solution, problem)
    greedy_fitness = calculate_aggregated_fitness(greedy_objectives)
    
    # Run multiple random trials
    random_fitnesses = []
    random_objectives_list = []
    for i in range(10):
        random_solution = random_baseline(problem)
        random_objectives = calculate_multi_objective_fitness(random_solution, problem)
        random_fitness = calculate_aggregated_fitness(random_objectives)
        random_fitnesses.append(random_fitness)
        random_objectives_list.append(random_objectives)
    
    avg_random_fitness = np.mean(random_fitnesses)
    avg_random_objectives = {
        key: np.mean([obj[key] for obj in random_objectives_list])
        for key in random_objectives_list[0].keys()
    }
    
    # GA results
    ga_objectives = ga_results['best_objectives']
    ga_fitness = ga_results['best_chromosome'].fitness
    
    # Performance comparison
    print("\nPerformance Comparison:")
    print("-" * 40)
    print(f"{'Method':<15} {'Fitness':<10} {'Cost':<10} {'Service':<10} {'Util':<10}")
    print("-" * 40)
    
    print(f"{'Random':<15} {avg_random_fitness:<10.4f} ${avg_random_objectives['total_cost']:<9.0f} "
          f"{avg_random_objectives['service_level']:<9.1f}% {avg_random_objectives['capacity_utilization']:<9.1f}%")
    
    print(f"{'Greedy':<15} {greedy_fitness:<10.4f} ${greedy_objectives['total_cost']:<9.0f} "
          f"{greedy_objectives['service_level']:<9.1f}% {greedy_objectives['capacity_utilization']:<9.1f}%")
    
    print(f"{'GA':<15} {ga_fitness:<10.4f} ${ga_objectives['total_cost']:<9.0f} "
          f"{ga_objectives['service_level']:<9.1f}% {ga_objectives['capacity_utilization']:<9.1f}%")
    
    # Improvement calculations
    cost_improvement_vs_greedy = ((greedy_objectives['total_cost'] - ga_objectives['total_cost']) / 
                                greedy_objectives['total_cost']) * 100
    service_improvement_vs_greedy = ga_objectives['service_level'] - greedy_objectives['service_level']
    
    cost_improvement_vs_random = ((avg_random_objectives['total_cost'] - ga_objectives['total_cost']) / 
                                 avg_random_objectives['total_cost']) * 100
    service_improvement_vs_random = ga_objectives['service_level'] - avg_random_objectives['service_level']
    
    print("\nImprovement Analysis:")
    print("-" * 30)
    print(f"GA vs Greedy:")
    print(f"  Cost reduction: {cost_improvement_vs_greedy:.1f}%")
    print(f"  Service improvement: {service_improvement_vs_greedy:.1f} percentage points")
    
    print(f"\nGA vs Random:")
    print(f"  Cost reduction: {cost_improvement_vs_random:.1f}%")
    print(f"  Service improvement: {service_improvement_vs_random:.1f} percentage points")
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    methods = ['Random', 'Greedy', 'GA']
    costs = [avg_random_objectives['total_cost'], greedy_objectives['total_cost'], ga_objectives['total_cost']]
    services = [avg_random_objectives['service_level'], greedy_objectives['service_level'], ga_objectives['service_level']]
    
    # Cost comparison
    bars1 = ax1.bar(methods, costs, color=['lightcoral', 'orange', 'green'])
    ax1.set_title('Total Cost Comparison')
    ax1.set_ylabel('Total Cost ($)')
    
    for bar, cost in zip(bars1, costs):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'${cost:.0f}', ha='center', va='bottom')
    
    # Service level comparison
    bars2 = ax2.bar(methods, services, color=['lightcoral', 'orange', 'green'])
    ax2.set_title('Service Level Comparison')
    ax2.set_ylabel('Service Level (%)')
    ax2.set_ylim(0, 100)
    
    for bar, service in zip(bars2, services):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{service:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'ga_objectives': ga_objectives,
        'greedy_objectives': greedy_objectives,
        'random_objectives': avg_random_objectives
    }

# Run comparison
comparison_results = compare_with_baselines()

### Why This Tier Exists vs Earlier Tiers

**Tier 3: Advanced Metaheuristic Optimization** - This tier provides sophisticated global optimization capabilities that can escape local optima and find near-optimal solutions for complex, multi-objective problems where traditional methods struggle.

**Key Advantages over Tier 1 and Tier 2:**
- **Global Search**: Explores diverse solution space to avoid local optima
- **Multi-Objective Optimization**: Simultaneously balances cost, service, and efficiency
- **Adaptive Learning**: Improves solution quality through evolutionary process
- **Scalability**: Handles complex interactions between many decision variables
- **Solution Diversity**: Maintains population of diverse solutions for robustness

**When to Use This Tier vs Earlier Tiers:**
- **Complex Trade-offs**: When multiple objectives conflict and require balancing
- **Large Search Spaces**: When problem complexity overwhelms exact methods
- **Non-Linear Relationships**: When objectives have complex, non-linear interactions
- **Robust Solutions Needed**: When solution reliability is critical

### Performance Comparison Summary

| Aspect | Tier 1 (MIP) | Tier 2 (Heuristic) | Tier 3 (GA) | When to Prefer |
|--------|-------------|-------------------|-------------|----------------|
| **Solution Quality** | Optimal | Good (85-95%) | Excellent (95-99%) | Tier 3 for complex multi-objective |
| **Computation Time** | Minutes-hours | Milliseconds-seconds | Seconds-minutes | Tier 2 for real-time, Tier 3 for quality |
| **Multi-Objective** | Limited | Single objective | Excellent | Tier 3 for conflicting objectives |
| **Global Optimum** | Guaranteed | Not guaranteed | High probability | Tier 3 for complex landscapes |
| **Adaptability** | Static | High | Medium | Tier 2 for dynamic, Tier 3 for optimization |
| **Implementation** | Complex | Simple | Medium | Tier 2 for simplicity, Tier 3 for power |

### Genetic Algorithm Innovations

**1. Multi-Objective Fitness Function:**
- Combines cost minimization, service maximization, capacity utilization, and load balancing
- Weighted aggregation allows priority adjustment based on business needs
- Normalization ensures comparable scales across objectives

**2. Adaptive Genetic Operators:**
- **Tournament Selection**: Maintains selection pressure while preserving diversity
- **Uniform Crossover**: Enables effective recombination of scheduling patterns
- **Adaptive Mutation**: Mutation strength adapts based on current gene values
- **Constraint Repair**: Ensures all solutions satisfy operational constraints

**3. Evolution Tracking and Analysis:**
- **Convergence Detection**: Identifies when further improvement becomes marginal
- **Diversity Monitoring**: Tracks population diversity to prevent premature convergence
- **Multi-Objective Evolution**: Shows how different objectives improve over generations

### Practical Applications

**Ideal Use Cases for Genetic Algorithm:**
- **Strategic Planning**: Long-term kitting strategy with multiple conflicting goals
- **Complex Constraints**: Problems with many interdependent constraints
- **Scenario Analysis**: Evaluating multiple what-if scenarios simultaneously
- **System Design**: Optimizing entire kitting system configuration

**Parameter Tuning Guidelines:**
- **Population Size**: 50-100 for medium problems, larger for complex instances
- **Generations**: 200-500 for convergence, monitor for early stopping
- **Crossover Rate**: 0.7-0.9 for balanced exploration/exploitation
- **Mutation Rate**: 0.05-0.15 for maintaining diversity
- **Elitism Ratio**: 0.05-0.15 for preserving best solutions

### Limitations and Mitigation

**Limitations:**
- **Computational Cost**: Higher than simple heuristics
- **Parameter Sensitivity**: Performance depends on parameter tuning
- **No Optimality Guarantee**: May not find true global optimum

**Mitigation Strategies:**
- **Hybrid Approaches**: Combine with local search for refinement
- **Parameter Adaptation**: Dynamic parameter adjustment during evolution
- **Multiple Runs**: Execute multiple times with different random seeds
- **Parallel Processing**: Leverage parallel evaluation for speedup

### Conclusion

The Genetic Algorithm tier provides powerful optimization capabilities that bridge the gap between fast heuristics and computationally intensive exact methods. By leveraging evolutionary principles, it can navigate complex solution spaces and balance multiple conflicting objectives effectively. While requiring more computational resources than simple heuristics, it delivers significantly better solution quality and can handle the complexity of real-world kitting operations with multiple interdependent constraints and objectives.

The GA's ability to maintain diverse solutions and adaptively explore the search space makes it particularly valuable for strategic kitting decisions where multiple performance metrics must be balanced simultaneously and solution robustness is critical.